# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mramir3/flyrank-intenship-ml/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

##Setup (Colab or local)

In [27]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "/content/flyrank-ml-internship-starter"  # absolute path -> re-running this cell can't nest

def has_data():
    return os.path.exists("data/raw/content_refresh_anonymized.csv")

if not has_data():
    if IN_COLAB:
        if not os.path.isdir(REPO_DIR):
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
        os.chdir(REPO_DIR)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    else:
        while not has_data() and os.getcwd() != "/":
            os.chdir("..")

print("Working dir:", os.getcwd())
assert has_data(), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

Working dir: /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter/flyrank-ml-internship-starter/flyrank-ml-internship-starter
Starter data found. You're ready.


## 1. My lane (or freestyle) and why

***Lane: Refresh / Content Opportunity Scoring.**

I'm picking this one because it's the lane the starter data and the whole baseline pipeline were literally built around — `content_refresh_anonymized.csv` is 30,000 pages with 90-day search and analytics history, and the pipeline already knows how to turn that into a ranked "fix this first" queue. That means I can spend Week 1 on framing the question honestly instead of fighting the data. The underlying decision — which stale or declining pages does an editor touch first — is also the most concrete of the four lanes: it ends in a ranked list a real person could act on this week, not just a report of interesting signals.*

In [28]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
print('rows, cols:', df.shape)
print('clients:', df['client_id'].nunique())
print(df['content_type'].value_counts())


rows, cols: (30000, 44)
clients: 32
content_type
keyword article       27207
feedly article         2096
comparison article      697
Name: count, dtype: int64


## 2. The question: decision, action, cost of a wrong call

***Decision:** out of thousands of existing pages, which ones should an editor spend this week's limited refresh time on?

**Who acts on it:** a content/SEO editor (or a small content team) working through a weekly refresh queue. The output they need is a short, ranked list of pages plus a reason for each one being on it — not a raw table of 30,000 rows.

**Cost of a wrong call, both directions:**
- **False positive** (flagged "refresh me" but the page was fine): wasted editor hours that could've gone to a page that actually needed it — a real opportunity cost, not a catastrophe.
- **False negative** (a genuinely declining page never surfaces): the page keeps losing impressions and clicks quietly, and since search visibility compounds (lower rankings feed fewer clicks feed weaker signals), a missed decline is more expensive to reverse the longer it sits unflagged.

Because a missed decline is costlier and slower to undo than a wasted edit, the queue should lean toward catching real declines even at the cost of a few unnecessary refreshes — precision matters, but recall on the truly declining pages matters more.*

In [29]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
declining = df['trend_direction'] == 'down'
print('pages currently declining:', declining.sum(), f'({declining.mean()*100:.1f}% of 30,000)')
print()
print('measurable_opportunity (>=100 impressions_90d AND sessions_90d>0):')
measurable = (df['impressions_90d'] >= 100) & (df['sessions_90d'] > 0)
print(measurable.sum(), f'({measurable.mean()*100:.1f}%) — the pool worth prioritizing at all')


pages currently declining: 16262 (54.2% of 30,000)

measurable_opportunity (>=100 impressions_90d AND sessions_90d>0):
22006 (73.4%) — the pool worth prioritizing at all


## 3. Quick look at the data (2-3 real numbers)

*Three numbers from the 30,000-page starter slice, computed below:

1. **54.2% of pages (16,262 / 30,000) are currently declining** (`trend_direction == 'down'`, impressions down >20% vs. the prior 30 days). That's the size of the problem an editor is facing — over half the site's existing content, at any moment, is losing search visibility.
2. **73.4% of pages (22,006 / 30,000) clear the "worth prioritizing" bar** — at least 100 impressions and at least one session in the last 90 days. So this isn't a needle-in-a-haystack problem; most of the catalog is a live candidate, which is exactly why editors need a ranked queue instead of eyeballing a spreadsheet.
3. **Decline rate rises with staleness**: pages last updated 91-180 days ago decline 61.1% of the time, vs. 51.1% for pages updated in the last 30 days — a 10-point gap. That's a real, observed pattern in the data (not proof staleness *causes* decline — see Section 4), and it's exactly the kind of signal a baseline rule or model could use to prioritize.*

In [30]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
n = len(df)

declining = df['trend_direction'] == 'down'
print(f"1. Declining now: {declining.sum():,} / {n:,} = {declining.mean()*100:.1f}%")

measurable = (df['impressions_90d'] >= 100) & (df['sessions_90d'] > 0)
print(f"2. Measurable opportunity: {measurable.sum():,} / {n:,} = {measurable.mean()*100:.1f}%")

decline_by_freshness = df.groupby('freshness_tier')['trend_direction'].apply(lambda s: (s == 'down').mean() * 100).round(1)
print("3. Decline rate by freshness_tier (days since last update):")
print(decline_by_freshness.sort_values(ascending=False))


1. Declining now: 16,262 / 30,000 = 54.2%
2. Measurable opportunity: 22,006 / 30,000 = 73.4%
3. Decline rate by freshness_tier (days since last update):
freshness_tier
91-180    61.1
31-90     58.9
0-30      51.1
181+      47.1
Name: trend_direction, dtype: float64


## 4. Careful words: what I can and can't claim

***What this work will be able to say:**
- *Observed:* on this 90-day slice, 54.2% of pages are declining, and staleness and decline rate move together.
- *Directional:* pages that haven't been updated in 91-180 days decline more often than recently-updated pages, by a measurable margin (61.1% vs 51.1%).
- *Decision-support:* a ranked queue that gives an editor a reasonable, evidence-backed place to start this week — a recommendation, not a verdict.

**What it will never say:**
- Not causal proof. Staleness correlating with decline doesn't mean refreshing a page *causes* it to recover — older pages may also cover topics that are naturally fading, sit in more competitive SERPs now, or simply be the pages nobody has gotten around to touching for other reasons. I haven't tested a before/after refresh here, only a snapshot.
- Not "predicting Google's algorithm." `trend_direction`/`trend_pct` are the *label* I'm trying to explain, not something I'm reverse-engineering from Google's ranking system — the data dictionary is explicit that these two columns are the label source and must never be used as features later on.
- Not a guarantee for any individual page. A ranked list surfaces good bets in aggregate; any single page can still be a false positive or false negative, which is why Section 2's cost framing matters going forward.*

In [31]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print(df['freshness_tier'].value_counts())
# 31-90 and 181+ are thin (175 and 174 rows) -- any claim built on those two tiers alone
# needs a bigger sample before it's trustworthy. The 0-30 vs 91-180 comparison (20,480 vs 9,171
# rows) is the one with enough volume to lean on this week.


freshness_tier
0-30      20480
91-180     9171
31-90       175
181+        174
Name: count, dtype: int64


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.